In [1]:
import xmlrpc.client as xml
import os
import csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import traceback

proxy = xml.ServerProxy("http://localhost:1080/RPC2")
plecs = proxy.plecs
directory = r"C:\Users\Goat_\Documents\Research\Class E\\"
model = "compression"
plecs.load(directory + model + ".plecs")

datadir = directory + model + "/data/"
os.makedirs(datadir, exist_ok=True)

In [2]:
def plot(name, file, axis, constant=1, multiply=None, col=None, close=None):
    data = pd.read_csv(directory + file)
    time = data['Time']  
    # maxcol = np.minimum(data.shape[0], data.shape[1])
    if multiply == True:
        plotpoint = data.iloc[:, 1] * data.iloc[:, 1]
        plt.plot(time * 1000000, plotpoint, label=col)
    elif col==None:
        for col in data.columns[1:]:
            plotpoint = data[col] * constant
            plt.plot(time * 1000000, data[col] * constant, label=col)
    else:
        plotpoint = data.iloc[:, col] * constant
        plt.plot(time * 1000000, data.iloc[:, col] * constant, label=col)

    plotlast = f"{plotpoint.iloc[-1]:.3f}"

    plt.text(time.iloc[-1], plotpoint.iloc[-1], f"{axis} {plotpoint.iloc[-1]:.3f}", 
         fontsize=10, color="red", ha="left", va="bottom")

    plt.xlabel("Time (us)")
    plt.ylabel(axis)
    plt.title(name)
    plt.legend()
    plt.grid(True)
    if close==None: 
        plt.savefig(datadir + name, dpi=100)
        plt.close()
    return plotlast

In [3]:
def efficiency(name, pin, inputV, pout, axis):
    pindata = pd.read_csv(directory + pin)
    poutdata = pd.read_csv(directory + pout)
    time = pindata['Time']
    eff = poutdata.iloc[:, 1] / (pindata.iloc[:, 1] * inputV * 2)
    plt.plot(time * 1e6, eff, label="%")
    last_val = float(eff.iloc[-1])
    plotlast = f"{last_val:.2f}"
    plt.text(time.iloc[-1] * 1e6, last_val, f"{axis} {last_val:.2f}", fontsize=10, color="red", ha="right", va="bottom")
    plt.xlabel("Time (us)")
    plt.ylabel(axis)
    plt.title(name)
    plt.legend()
    plt.grid(True)
    plt.savefig(datadir + name, dpi=100)
    plt.close()
    return plotlast


In [4]:
def get_data(file):
    data = pd.read_csv(directory + file)
    return data

In [5]:
freq = np.linspace(800e3, 1200e3, num=21, endpoint=True)
print("Freq:", freq)

# pwm = np.linspace(0.2, 0.8, num=13, endpoint=True)
# print("PWM:", pwm)

loads = np.array([0, 5000, 10000, 15000, 20000, 25000, 30000, 50000, 70000, 100000])
print("Loads:", loads)

Freq: [ 800000.  820000.  840000.  860000.  880000.  900000.  920000.  940000.
  960000.  980000. 1000000. 1020000. 1040000. 1060000. 1080000. 1100000.
 1120000. 1140000. 1160000. 1180000. 1200000.]
Loads: [     0   5000  10000  15000  20000  25000  30000  50000  70000 100000]


In [6]:
filename = datadir + "data.csv"
if not os.path.exists(filename):
    with open(filename, "w") as f:
        pass

In [ ]:
curr_load = int(input())
j = 0

while curr_load < len(loads):
    datadir = f"{directory}{model}/data/R-{loads[curr_load]/1e3:.1f}kohm/"
    os.makedirs(datadir, exist_ok=True)
    while j < len(freq):
        current_test = f"F-{freq[j]/1e3:.1f}_kHz_R-{loads[curr_load]/1e3:.1f}-kOhm ({j} + {curr_load})"
        print(f"\rF: {freq[j]/1e3:.1f} kHz, R: {loads[curr_load]/1e3:.1f} kOhm", end="", flush=True)
        opts = {'ModelVars' : { 
            'rload' : float(loads[curr_load]),
            'freq' : float(freq[j]),
        } }
        while True:
            try:
                # code that might fail
                plecs.simulate(model, opts)
                pin  = plot(os.path.join(current_test + "_PIN.png"),  "PIN.csv",  "PIN", 6, None, None, False)
                pout = plot(os.path.join(current_test + "_POWER.png"), "POUT.csv", "Power")
                zvs  = plot(os.path.join(current_test + "_ZVS.png"),  "ZVS.csv",  "Voltage", 1, None, None, False)
                eff  = efficiency(os.path.join(current_test + "_ZVS-EFFICIENCY.png"), "PIN.csv", 3, "POUT.csv", "Efficiency")
                vout = plot(os.path.join(current_test + "_VOUT.png"), "OUTPUT.csv", "Voltage", 1, None, 1)
                iout = plot(os.path.join(current_test + "_IOUT.png"), "OUTPUT.csv", "Current", 1, None, 2)
                write_to_csv = [pin, pout, zvs, vout, iout, eff, freq[j]]
                with open(filename, "a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow(write_to_csv)
                plt.close('all')
                j += 1
                break
            except Exception as e:
                print(e)
                skip = input("sim failed - skip?")
                if skip == "y":
                    j += 1
                    break
                if type(skip) == int:
                    j += int(skip)
                    break
    curr_load += 1

 2


F: 800.0 kHz, R: 10.0 kOhm

# MANUAL MODE

In [19]:
curr_load = int(input())
j = int(input())
print(f"\rF: {freq[j]/1e3:.1f} kHz, R: {loads[curr_load]/1e3:.1f} kOhm", end="", flush=True)

 2
 20


F: 1200.0 kHz, R: 10.0 kOhm

In [21]:
opts = {'ModelVars' : { 
    'rload' : float(loads[curr_load]),
    'freq' : float(freq[j]),
} }
plecs.simulate(model, opts)

Fault: <Fault -34000: "Simulation error: Error: Cyclic behavior during switching.\n\nSwitches that bounced:\n- 'compression/Multiplier1/D1'\n">

In [22]:
datadir = f"{directory}{model}/data/R-{loads[curr_load]/1e3:.1f}kohm/"
os.makedirs(datadir, exist_ok=True)
current_test = f"F-{freq[j]/1e3:.1f}_kHz_R-{loads[curr_load]/1e3:.1f}-kOhm ({j} + {curr_load})"
print(f"\rF: {freq[j]/1e3:.1f} kHz, R: {loads[curr_load]/1e3:.1f} kOhm", end="", flush=True)

try:
    pin  = plot(os.path.join(current_test + "_PIN.png"),  "PIN.csv",  "PIN", 6, None, None, False)
    pout = plot(os.path.join(current_test + "_POWER.png"), "POUT.csv", "Power")
    zvs  = plot(os.path.join(current_test + "_ZVS.png"),  "ZVS.csv",  "Voltage", 1, None, None, False)
    eff  = efficiency(os.path.join(current_test + "_ZVS-EFFICIENCY.png"), "PIN.csv", 3, "POUT.csv", "Efficiency")
    vout = plot(os.path.join(current_test + "_VOUT.png"), "OUTPUT.csv", "Voltage", 1, None, 1)
    iout = plot(os.path.join(current_test + "_IOUT.png"), "OUTPUT.csv", "Current", 1, None, 2)
    write_to_csv = [pin, pout, zvs, vout, iout, eff, freq[j]]
    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(write_to_csv)
    plt.close('all')
except Exception as e:
    print(e)
    traceback.print_exc()

F: 1200.0 kHz, R: 10.0 kOhm

## Testing

In [ ]:
plot("PIN", "PIN.csv", "Power", 6)

In [ ]:
plot("POUT", "POUT.csv", "Power")

In [ ]:
plot("ZVS", "ZVS.csv", "Voltage")

In [ ]:
plot("VOUT", "OUTPUT.csv", "Voltage", 1, None, 1)

In [ ]:
plot("IOUT", "OUTPUT.csv", "Current", 1, None, 2)

In [ ]:
efficiency("EFFICIENCY", "PIN.csv", 3, "POUT.csv", "Efficiency")

In [ ]:
plecs.simulate(model)

In [ ]:
opts = {'ModelVars' : { 
    'rload' : 15000,
    'freq' : 2e6,
    'pwm' : 30
} }
plecs.simulate(model, opts)